In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers>=0.0.27" "trl==0.9.6" peft accelerate bitsandbytes

In [ ]:
import os

# Đường dẫn file gây lỗi trong thư viện Unsloth
file_path = "/usr/local/lib/python3.12/dist-packages/unsloth/models/rl.py"

if os.path.exists(file_path):
    print("🛠️ Đang tiến hành vá lỗi KeyError trong rl.py...")

    with open(file_path, "r") as f:
        code = f.read()

    # Tìm và comment dòng lệnh gây chết chương trình
    # Lỗi do Unsloth cố tìm key "sanitize_logprob" nhưng không thấy
    bad_line = 'sanitize_logprob = RL_REPLACEMENTS["sanitize_logprob"]'
    patched_line = 'sanitize_logprob = None # Đã vá lỗi bởi Gemini Nuclear Option'

    if bad_line in code:
        new_code = code.replace(bad_line, patched_line)
        with open(file_path, "w") as f:
            f.write(new_code)
        print("✅ Đã vá thành công! Dòng lệnh gây lỗi đã bị vô hiệu hóa.")
    else:
        print("⚠️ Không tìm thấy dòng lỗi. Có thể thư viện đã thay đổi hoặc bạn chưa cài đúng bản.")
else:
    print("❌ Không tìm thấy file thư viện. Hãy chắc chắn Bước 2 đã chạy xong.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-3B-Instruct",
    load_in_4bit = True,
)
print("✅ Hệ thống đã sẵn sàng để train LoRA!")

In [ ]:
from unsloth import FastLanguageModel
import torch
from trl import SFTTrainer, SFTConfig
import transformers
import wandb
from datasets import load_dataset

# =================================================================
# 1. KHỞI TẠO WANDB (HỆ THỐNG GIÁM SÁT)
# =================================================================
wandb.login()
wandb.init(
    project="Qwen-Gaming-Translator",
    name="Run-V1-160326-Gaming", # Đặt tên mới để phân biệt với các lần chạy trước
    tags=["gaming", "translation", "qwen-2.5-3b"]
)

# =================================================================
# 2. CẤU HÌNH CƠ BẢN VÀ MODEL
# =================================================================
max_seq_length = 2048 # T4 16GB dư sức chạy 2048
dtype = None # Auto detection
load_in_4bit = True # Bắt buộc để tiết kiệm VRAM

print("⏳ Đang tải Model Base...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-3B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

print("⏳ Đang thiết lập LoRA Adapter...")
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank cao (16) để học sâu
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16, # Hệ số alpha = r giúp bảo toàn kiến thức gốc
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# =================================================================
# 3. CHUẨN BỊ VÀ PHÂN LOẠI DỮ LIỆU (TRAIN / EVAL)
# =================================================================
alpaca_prompt = """<|im_start|>system
Bạn là một trợ lý dịch thuật chuyên nghiệp.<|im_end|>
<|im_start|>user
{instruction}
{input}<|im_end|>
<|im_start|>assistant
{output}<|im_end|>"""

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction=instruction, input=input, output=output)
        texts.append(text)
    return { "text" : texts, }

print("⏳ Đang phân loại Dataset...")
# Nạp tập Train (Quét mọi file có cụm '-train-' ở thư mục gốc và thư mục con)
train_dataset = load_dataset(
    "json",
    data_files=["*-train-*.jsonl", "*/*-train-*.jsonl"],
    split="train"
)
train_dataset = train_dataset.map(formatting_prompts_func, batched = True)
print(f"✅ Đã nạp {len(train_dataset)} câu vào tập TRAIN.")

# Nạp tập Eval (Quét mọi file có cụm '-eval-' ở thư mục gốc và thư mục con)
try:
    eval_dataset = load_dataset(
        "json",
        data_files=["*-eval-*.jsonl", "*/*-eval-*.jsonl"],
        split="train"
    )
    eval_dataset = eval_dataset.map(formatting_prompts_func, batched = True)
    has_eval = True
    print(f"✅ Đã nạp {len(eval_dataset)} câu vào tập EVAL.")
except ValueError:
    print("⚠️ KHÔNG TÌM THẤY TẬP EVAL. Hệ thống sẽ bỏ qua bước đánh giá và chỉ chạy Train.")
    has_eval = False

# =================================================================
# 🛠️ MAGIC PATCH: "Vá nóng" thư viện Hugging Face
# =================================================================
if not hasattr(transformers.Trainer, "_is_patched_for_tokenizer"):
    original_init = transformers.Trainer.__init__
    def new_init(self, *args, **kwargs):
        if "tokenizer" in kwargs:
            kwargs["processing_class"] = kwargs.pop("tokenizer")
        original_init(self, *args, **kwargs)
    transformers.Trainer.__init__ = new_init
    transformers.Trainer._is_patched_for_tokenizer = True

# =================================================================
# 4. CẤU HÌNH TRAINING VÀ BÁO CÁO WANDB
# =================================================================
print("⏳ Đang thiết lập cấu hình huấn luyện...")
training_args = SFTConfig(
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_steps = 5,
    max_steps = 100,
    learning_rate = 2e-4,
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    logging_steps = 1,
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "linear",
    seed = 3407,
    output_dir = "outputs",

    # Bật tính năng báo cáo WandB và thiết lập tần suất chấm điểm Eval
    report_to = "wandb",
    eval_strategy = "steps" if has_eval else "no",
    eval_steps = 10 if has_eval else None, # Cứ 10 step thì giải đề Eval 1 lần
    per_device_eval_batch_size = 2 if has_eval else None,
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset if has_eval else None,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = training_args,
)

# =================================================================
# 5. KHỞI CHẠY TRAINING
# =================================================================
print("🚀 Đang khởi động tiến trình Train...")
transformers.logging.set_verbosity_error()
trainer_stats = trainer.train()

print("✅ QUÁ TRÌNH HUẤN LUYỆN HOÀN TẤT!")
wandb.finish()
print("📈 Đã đóng kết nối WandB. Vui lòng kiểm tra biểu đồ trên Dashboard.")

In [ ]:
# Lưu vào thư mục trên Colab
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")

# Nén lại để tải về cho dễ
!zip -r lora_model.zip lora_model

# Tải về máy tính
from google.colab import files
files.download('lora_model.zip')

In [ ]:
# 1. Định nghĩa lại Prompt "Mở" (Không có <|im_end|> ở cuối)
inference_prompt = """<|im_start|>system
Bạn là một trợ lý dịch thuật chuyên nghiệp.<|im_end|>
<|im_start|>user
{instruction}
{input}<|im_end|>
<|im_start|>assistant
""" # <--- DỪNG Ở ĐÂY, để model tự viết tiếp

# 2. Chạy Inference
FastLanguageModel.for_inference(model)
inputs = tokenizer(
[
    inference_prompt.format(
        instruction="Dịch thuật ngữ cảnh Gaming",
        input="Quiet everyday real terror. Each episode reconstructs the account of a survivor, ordinary people who went through extreme situations and chose to share their stories. So, this is based off of True Story, too. We played the first one a few months ago. Y'all remember it? Of course you do. That's a crazy name.",
    )
], return_tensors = "pt").to("cuda")

# 3. Generate
outputs = model.generate(**inputs, max_new_tokens = 128, use_cache = True)

# 4. In kết quả (Lọc bỏ phần prompt thừa)
result = tokenizer.batch_decode(outputs)[0]
# Lấy phần sau thẻ assistant cuối cùng
clean_result = result.split("<|im_start|>assistant")[-1].replace("<|im_end|>", "").strip()

print("Kết quả AI dịch:")
print(clean_result)

In [ ]:
# Convert sang GGUF (Q4_K_M là chuẩn cân bằng nhất)
model.save_pretrained_gguf("model_gguf", tokenizer, quantization_method = "q4_k_m")

# Mount Drive (nếu chưa) để copy file GGUF to đùng (~2GB) vào Drive cho nhanh
!cp model_gguf/Qwen2.5-3B-Instruct-unsloth.Q4_K_M.gguf /content/drive/MyDrive/